# Practice 3 - TV5: Training and Fine-tuning

## Responsibility
Define training arguments, create `Trainer`, and fine-tune the pretrained model for binary text classification.

## Sections
- Import training utilities
- Define training arguments
- Build `Trainer`
- Fine-tune pretrained model
- Save relevant training results/checkpoints if required


## 1. Import libraries và cấu hình chung

In [1]:
import torch
import numpy as np
import evaluate
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# Thiết lập thiết bị (GPU nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị: {device}")

q:\Deep_Learning\UTH-Deep-Learning-nhom2\.venv-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sử dụng thiết bị: cuda


## 2. Load Processed Dataset

In [2]:
# Load dataset từ thư mục Văn Sơn đã bàn giao
dataset_path = "processed_imdb_dataset"
processed_dataset = load_from_disk(dataset_path)

# Lấy các tập train và validation để huấn luyện
train_dataset = processed_dataset["train"]
eval_dataset = processed_dataset["validation"]

print("Số lượng mẫu Train:", len(train_dataset))
print("Số lượng mẫu Validation:", len(eval_dataset))

Số lượng mẫu Train: 22500
Số lượng mẫu Validation: 2500


## 3. Load Pretrained Model & Tokenizer

In [3]:
MODEL_NAME = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(dataset_path + "/tokenizer") 
# (Lưu ý: Sơn có lưu tokenizer vào processed_imdb_dataset/tokenizer)

# Load model với cấu hình 2 nhãn đã thống nhất
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Negative", 1: "Positive"},
    label2id={"Negative": 0, "Positive": 1}
)
model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4353.29it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

## 4. Thiết Lập Hàm Tính Toán Metrics
Để Trainer theo dõi được độ chính xác trong quá trình huấn luyện, cần định nghĩa một hàm đánh giá (metric). Thư viện evaluate của Hugging Face sinh ra để làm việc này.

In [4]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Logits chứa xác suất chưa chuẩn hóa, ta lấy argmax để tìm nhãn dự đoán
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## 5. Define Training Arguments & Create Trainer
Trainer mặc định sử dụng AdamW kết hợp với Weight Decay để tránh Overfitting.

In [9]:
# Cấu hình tham số huấn luyện (Training Arguments)
training_args = TrainingArguments(
    output_dir="./imdb_finetuned_results", # Thư mục lưu checkpoint và log
    num_train_epochs=3,                    # Số epoch huấn luyện toàn bộ dữ liệu
    per_device_train_batch_size=16,        # Batch size cho train
    per_device_eval_batch_size=16,         # Batch size cho eval
    learning_rate=2e-5,                    # Tốc độ học nhỏ vì đang fine-tune
    weight_decay=0.01,                     # L2 Regularization
    eval_strategy="epoch",                 # Đánh giá sau mỗi epoch
    save_strategy="epoch",                 # Lưu checkpoint sau mỗi epoch
    load_best_model_at_end=True,           # Lấy mô hình có val_loss thấp nhất
    logging_steps=100,                     # In log mỗi 100 bước
)

# Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

## 6. Start Training & Save Model

In [10]:
# Bắt đầu quá trình huấn luyện
print("Tiến hành Fine-tuning mô hình trên tập IMDB...")
trainer.train()

# Bàn giao cho Thành Thi (Evaluation)
final_model_path = "./fine_tuned_imdb_distilbert"
trainer.save_model(final_model_path)
print(f"Huấn luyện hoàn tất! Mô hình đã được lưu tại: {final_model_path}")

Tiến hành Fine-tuning mô hình trên tập IMDB...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.243846,0.256738,0.904400
2,0.175339,0.312637,0.907600
3,0.121371,0.369939,0.914800


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]

Huấn luyện hoàn tất! Mô hình đã được lưu tại: ./fine_tuned_imdb_distilbert


## Handover
Record the training configuration, training progress, and final model/checkpoint information for TV6.